# Capa Silver - Catálogo de Empleados

In [1]:
from spark_session import *
from transformation_functions import *
from delta.tables import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, max, col

## SparkSession

In [2]:
spark = create_spark_session(
    "retail-dlh-silver",
    configs=[
        configure_delta_extensions,
        configure_delta_catalog,
        configure_s3
    ]
)

## Extracción

In [3]:
# - Obtiene dataframe de la capa bronze
df_employee = spark.table("bronze.employee")

df_employee = df_employee.drop(col("REGION"))
df_employee = df_employee.withColumn("EMPLOYEE_ID", col("EMPLOYEE_ID").cast("decimal(6, 0)"))

df_salaries = spark.table("bronze.salaries")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Transformaciones

In [4]:
# - Filtra los empleados validos
df_employee_filter = df_employee.where(
     df_employee["EMPLOYEE_ID"].isNotNull()
)

# - Asignación de consecutivo por empleado
df_consecutivo = df_employee_filter.withColumn("CONSECUTIVO", row_number().over(
        Window.partitionBy(col("EMPLOYEE_ID")).orderBy(col("DATE_ENTERED"))
    )
)

# - Obtiene el maximo consecutivo por empleado
df_employee_max = df_consecutivo.groupBy("EMPLOYEE_ID").agg(
    max(col("CONSECUTIVO")).alias("MAX_CONSECUTIVO")
).withColumnRenamed("EMPLOYEE_ID", "MAX_EMPLOYEE_ID")

# - Obtiene el empleado unico, nombre completo, formato del telefono, genero completo, grupo de edad
df_join_employee = df_consecutivo.join(
    df_employee_max, 
    ((df_consecutivo.EMPLOYEE_ID == df_employee_max.MAX_EMPLOYEE_ID) & 
     (df_consecutivo.CONSECUTIVO == df_employee_max.MAX_CONSECUTIVO)),
    "inner"
).withColumn("EMPLOYEE_NAME", 
             combinar_nombre_apellido_udf(col("FIRST_NAME"), col("LAST_NAME"))
).withColumn("EMPLOYEE_PHONE_NMBR", 
             formato_telefono_udf(col("PHONE_NUMBER"))
).withColumn("EMPLOYEE_GENDER",
             genero_completo_udf(col("GENDER"))
).withColumn("AGE_GROUP", 
             grupo_edad_udf(col("AGE"))
)

# - Renombra columnas
df_employees = df_join_employee.select(
    col("EMPLOYEE_ID"),
    col("EMPLOYEE_NAME"),
    col("ADDRESS").alias("EMPLOYEE_ADDRESS"),
    col("CITY").alias("EMPLOYEE_CITY"),
    col("STATE").alias("EMPLOYEE_STATE"),
    col("ZIP_CODE").alias("EMPLOYEE_ZIP_CODE"),
    col("COUNTRY").alias("EMPLOYEE_COUNTRY"),
    col("EMPLOYEE_PHONE_NMBR"),
    col("FAX_NUMBER").alias("EMPLOYEE_FAX_NMBR"),
    col("EMAIL").alias("EMPLOYEE_EMAIL"),
    col("EMPLOYEE_GENDER"),    
    col("AGE_GROUP"),
    col("NATIVE_LANGUAGE").alias("NATIVE_LANG_DESC"),
    col("SECOND_LANGUAGE").alias("SEC_LANG_DESC"),
    col("THIRD_LANGUAGE").alias("TER_LANG_DESC"),
    col("POSITION_TYPE"),
    col("DEALERSHIP_ID"),
    col("REGIONAL_MANAGER"),
    col("HIRE_DATE"),
    col("DATE_ENTERED")
)

df_managers = df_employees.where("POSITION_TYPE = 'MANAGER'").select(
    col("DEALERSHIP_ID"),
    col("EMPLOYEE_NAME").alias("DEALERSHIP_MANAGER")
)

df_employees_final = df_employees.join(
    df_managers, (df_employees.DEALERSHIP_ID == df_managers.DEALERSHIP_ID), "inner"
).join(
    df_salaries, (df_employees.EMPLOYEE_ID == df_salaries.EMPLOYEE_ID), "left_outer"
).select (df_employees["*"],
          df_managers["DEALERSHIP_MANAGER"],
          df_salaries["SALARY"]
).withColumnRenamed("SALARY", "EMPLOYEE_SALARY")

## Carga

In [5]:
if DeltaTable.isDeltaTable(spark, "silver.employees"):
  # - Obtiene tabla delta de la capa silver
  deltaTblSilEmployees = DeltaTable.forName(spark, "silver.employees")

  ## --Inserta nuevos regitros, actualiza registros con cambios
  deltaTblSilEmployees.alias('SILV') \
    .merge(
      df_employees_final.alias('BRON'),
      condition='BRON.EMPLOYEE_ID = SILV.EMPLOYEE_ID'
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
else:
  ## --Inserta registros por primera vez
  df_employees_final.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.employees")

## Vacuum

In [6]:
# - Actualiza la configuración para que pueda establecer un período de retención cero.
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

#- Ejecutemos el comando de VACUUM 
#spark.sql(f"VACUUM '{vPathSilv}EMPLOYEES.delta' RETAIN 0 HOURS").show(truncate=False)

## Cierre de SparkSession

In [7]:
spark.stop()